# 03_finetune_stage2 — Format-Only Fine-Tuning on CREST

Trains a **separate** LoRA adapter on CREST guidelines to teach BioMistral the
output shape and citation habit.

**Limitation**: This adapter learns *format and citation fidelity*, not clinical
correctness. It is trained on synthetic (query, retrieved-snippets) → recommendation
pairs derived purely from CREST rows; no MIMIC patient labels are used.
The Stage 1 adapter (clinical reasoning) and Stage 2 adapter (citation format)
are loaded independently in `04_pipeline.ipynb`.

**Input** per example: a retrieval query derived from the guideline title + the
guideline's own text as a retrieved snippet + 2 random distractor snippets.
**Target**: the guideline text followed by `— {developer} ({strength})`.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q --no-cache-dir \
    torchvision==0.19.0 \
    torchaudio==2.4.0 \
    transformers==4.51.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    bitsandbytes>=0.46.1

In [2]:
import os
import pickle
from kaggle_secrets import UserSecretsClient
import huggingface_hub

# Paths
WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# HuggingFace login
secrets = UserSecretsClient()
huggingface_hub.login(
    token=secrets.get_secret("HF_TOKEN"),
    add_to_git_credential=False
)

In [3]:
import sys
sys.path.append("/kaggle/input/datasets/aanyagupta0921/senior-project-utils")

from utils import (
    load_biomistral_4bit,
    format_stage1_prompt,
    format_stage2_prompt,
    build_rag_index,
    retrieve,
    render_patient_note,
    _STAGE2_INSTRUCTION
)

In [4]:
import pandas as pd

crest_df = pd.read_csv('/kaggle/input/datasets/aanyagupta0921/senior-project-crest-dataset/crest_data.csv')
crest_df = crest_df.dropna(subset=["title", "developer", "text", "recommendation_norm"]).reset_index(drop=True)
print(f"Loaded {len(crest_df)} CREST rows")
print(f"Text length  — mean: {int(crest_df['text'].str.len().mean())}  max: {int(crest_df['text'].str.len().max())} chars")
crest_df[["title", "developer", "recommendation_norm"]].head(3)


Loaded 1604 CREST rows
Text length  — mean: 200  max: 1629 chars


,title,developer,recommendation_norm
0,Establishing the diagnosis of lung cancer: dia...,American College of Chest Physicians,strong
1,Establishing the diagnosis of lung cancer: dia...,American College of Chest Physicians,strong
2,Establishing the diagnosis of lung cancer: dia...,American College of Chest Physicians,strong


In [5]:
import re
import numpy as np

N_DISTRACTORS = 2
RANDOM_SEED   = 42
rng = np.random.default_rng(RANDOM_SEED)


def _paraphrase_title(title):
    """Deterministically derive a retrieval-query string from a CREST title."""
    segment = re.split(r":", title)[0].strip()
    segment = re.sub(r"\b\d+(st|nd|rd|th)\s+(ed(ition)?)\b", "", segment, flags=re.IGNORECASE)
    segment = re.sub(r"\b(19|20)\d{2}\b", "", segment)
    segment = re.sub(r"\s+", " ", segment).strip(" ,.")
    return f"guidelines for {segment.lower()}" if segment else f"guidelines for {title[:50].lower()}"


def _make_stage2_prompt(query, snippets, target):
    snippet_blocks = [
        f"[{i+1}] {s['title']} ({s['developer']}, {s['recommendation_norm']}):\n{s['text']}"
        for i, s in enumerate(snippets)
    ]
    user_content = (
        f"{_STAGE2_INSTRUCTION}\n\n"
        f"Retrieval query: {query}\n\n"
        f"Retrieved Guidelines:\n" + "\n\n".join(snippet_blocks)
    )
    return f"<s>[INST] {user_content} [/INST] {target} </s>"


def make_stage2_example(idx):
    row   = crest_df.iloc[idx]
    query = _paraphrase_title(row["title"])

    # 2 distractor indices drawn without replacement from the rest of the rows
    other_idxs      = [i for i in range(len(crest_df)) if i != idx]
    distractor_idxs = rng.choice(other_idxs, size=N_DISTRACTORS, replace=False).tolist()

    # Shuffle so the target snippet is not always in a fixed position
    snippet_idxs = [idx] + distractor_idxs
    rng.shuffle(snippet_idxs)

    snippets = [
        {
            "title":               str(crest_df.iloc[i]["title"]),
            "developer":           str(crest_df.iloc[i]["developer"]),
            "text":                str(crest_df.iloc[i]["text"]),
            "recommendation_norm": str(crest_df.iloc[i]["recommendation_norm"]),
        }
        for i in snippet_idxs
    ]

    target = f"{row['text']} \u2014 {row['developer']} ({row['recommendation_norm']})"
    return _make_stage2_prompt(query, snippets, target)


all_texts = [make_stage2_example(i) for i in range(len(crest_df))]
print(f"Built {len(all_texts)} Stage 2 examples")

# Token-length sanity check (chars / 4 ≈ tokens)
token_est = [len(t) // 4 for t in all_texts]
print(f"Estimated tokens — min: {min(token_est)}  max: {max(token_est)}  mean: {sum(token_est)//len(token_est)}")

print("\n=== Sample example===")
print(all_texts[0])


Built 1604 Stage 2 examples
Estimated tokens — min: 292  max: 1223  mean: 448

=== Sample example===
<s>[INST] You are a clinical decision support assistant. Given a patient record, predicted diagnoses, and retrieved clinical guidelines, write a treatment recommendation. End each suggestion with a citation in the form: — Developer (strength).

Retrieval query: guidelines for establishing the diagnosis of lung cancer

Retrieved Guidelines:
[1] Symptom management in patients with lung cancer: diagnosis and management of lung cancer, 3rd ed: American College of Chest Physicians evidence-based clinical practice guidelines. (American College of Chest Physicians, strong):
In lung cancer patients with psychologic symptoms, a comprehensive symptom management plan is recommended. This should include non-pharmacologic interventions integrated with medication management, which may be offered as a single treatment modality.

[2] Establishing the diagnosis of lung cancer: diagnosis and management o

In [6]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_texts, val_texts = train_test_split(all_texts, test_size=0.1, random_state=42)
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset   = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")


Train: 1443  Val: 161


In [7]:
model, tokenizer = load_biomistral_4bit()
print("Model loaded.")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

2026-05-08 00:55:11.717250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778201711.972623      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778201712.047859      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778201712.642910      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778201712.642966      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778201712.642970      22 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded.


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 27,262,976 || all params: 7,268,995,072 || trainable%: 0.3751


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

STAGE2_ADAPTER_PATH = "/kaggle/working/stage2_adapter"

training_args = TrainingArguments(
    output_dir=STAGE2_ADAPTER_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    max_seq_length=3072,
    dataset_text_field="text",
)

trainer.train()

trainer.model.save_pretrained(STAGE2_ADAPTER_PATH)
tokenizer.save_pretrained(STAGE2_ADAPTER_PATH)
print(f"Stage 2 adapter saved to {STAGE2_ADAPTER_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1443 [00:00<?, ? examples/s]

Map:   0%|          | 0/161 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:412: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/

Epoch,Training Loss,Validation Loss
1,0.389500,0.352800
2,0.104700,0.169241


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an except

Stage 2 adapter saved to /kaggle/working/stage2_adapter
